# Data vs MC — cumulative exposure animation

Animated GIF showing how **statistical error bars shrink** as beam data accumulate in
chronological order.

- **Data slicing**: 15 cumulative time-ordered steps (`beam_quality.ipynb` logic:
  sort `hdr` by `run`, `evt`, split into 15 chunks; step *k* uses chunks `0..k-1`).
- **Plotting**: `overlay_hists` / `data_vs_mc_plotter` from `data_mc_comparison.ipynb`.
- **Time axis**: each frame includes a progress bar below the plot, filled in sync with
  the cumulative beam `global_trigger_time` range.
- **Output**: two GIFs per variable — fixed y-axis (final-step scale) and auto y-axis
  (rescaled each frame).


In [27]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import tempfile
from datetime import datetime
from functools import partial
from os import makedirs, path
from pathlib import Path

import imageio.v3 as iio
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from PIL import Image

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')
from pyanalib.split_df_helpers import load_dfs
from pyanalib.split_df_helpers_new import dfs_from_dir
from pyanalib.variable_calculator import get_cc1p0pi_tki
from pyanalib.pandas_helpers import pad_column_name

from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi import utils as numucc_utils
from analysis_village.numucc_1p0pi.files_config import *
from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE
from analysis_village.numucc_1p0pi.syst_disk_layout import category_summary_npz_path

plt.style.use("presentation.mplstyle")
numucc_utils.fig_ext = ".png"


In [29]:
# --- paths & animation settings ---
DFS_ROOT = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs"
QUALITY_DF_DIR = path.join(DFS_ROOT, "2026_05_16_230705__sel_mup-data-1e20/merged_perTPC")
QUALITY_DF_PATH = path.join(QUALITY_DF_DIR, "beam_data_1e20_qualitycut.df")

DIR_MC = path.join(DFS_ROOT, "2026_05_11_084007__sel_mup-wgts_mcstat/merged_perTPC")
DIR_INTIME = path.join(DFS_ROOT, "2026_05_11_083624__sel_mup-mc-Intime/merged_perTPC")
DIR_OFFBEAM = path.join(DFS_ROOT, "2026_05_11_040015__sel_mup-data-OffBeamLight/merged_perTPC")

KEYS2LOAD = ["hdr", "evt"]
N_MAX_CONCAT = 999

FOM_POT_SCALE = 0.9822
INTIME_FRACTION = 0.08

N_TIME_SPLITS = 15
GIF_FRAME_MS = 500  # milliseconds per frame
GIF_FPS = int(round(1000 / GIF_FRAME_MS))

AX_YLIM_RATIO = 1.9
BREAKDOWN_TYPE = "topology"

var_configs = [
    VariableConfig.tki_del_alpha(),
]

LEGEND_PERCENTAGES_OVERRIDE = {
    "topology": [91.2, 3.1, 2.9, 1.5, 0.3, 1.1][::-1],
}

today_str = datetime.now().strftime("%Y%m%d")
gif_out_dir = path.join(save_fig_base_dir, "gifs", f"data_mc_comparison_fancyplot_{today_str}")
makedirs(gif_out_dir, exist_ok=True)

print("quality-cut df:", QUALITY_DF_PATH)
print("gif dir:", gif_out_dir)


quality-cut df: /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_16_230705__sel_mup-data-1e20/merged_perTPC/beam_data_1e20_qualitycut.df
gif dir: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/gifs/data_mc_comparison_fancyplot_20260611


## Load MC, intime, off-beam, and beam-quality data


In [4]:
mc_dfs = dfs_from_dir(DIR_MC, filename_str="sel_mup-wgts_mcstat", keys2load=KEYS2LOAD, n_max_concat=N_MAX_CONCAT)
intime_dfs = dfs_from_dir(DIR_INTIME, filename_str="sel_mup-mc-Intime", keys2load=KEYS2LOAD, n_max_concat=N_MAX_CONCAT)
offbeam_dfs = dfs_from_dir(DIR_OFFBEAM, filename_str="sel_mup-data-OffBeamLight", keys2load=KEYS2LOAD, n_max_concat=N_MAX_CONCAT)

mc_evt_df = mc_dfs["evt"]
mc_hdr_df = mc_dfs["hdr"]
intime_evt_df = intime_dfs["evt"]
intime_hdr_df = intime_dfs["hdr"]
offbeam_hdr_df = offbeam_dfs["hdr"]

mc_evt_df.loc[mc_evt_df.mc.iscc.isna(), ("mc", "iscc")] = 999

quality_dfs = load_dfs(QUALITY_DF_PATH, keys2load=["hdr", "trigger", "evt_good"], n_max_concat=1)
data_evt_df = quality_dfs["evt_good"]
data_hdr_df = quality_dfs["hdr"].join(quality_dfs["trigger"])
data_evt_df[("mc", "iscc")] = 999

mc_tot_pot = mc_hdr_df["pot"].sum()
intime_gates = offbeam_hdr_df[offbeam_hdr_df["first_in_subrun"] == 1]["noffbeambnb"].sum()

print(f"evt_good rows: {len(data_evt_df):,}")
print(f"mc evt: {len(mc_evt_df):,}  intime evt: {len(intime_evt_df):,}")


Found 2 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_084007__sel_mup-wgts_mcstat/merged_perTPC/2026_05_11_084007__sel_mup-wgts_mcstat_merged_0000.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_084007__sel_mup-wgts_mcstat/merged_perTPC/2026_05_11_084007__sel_mup-wgts_mcstat_merged_0001.df']


100%|██████████| 2/2 [00:18<00:00,  9.46s/it]


REMEMBER TO RECALCULATE TKI AND CHECK FV!!
Found 1 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_083624__sel_mup-mc-Intime/merged_perTPC/2026_05_11_083624__sel_mup-mc-Intime_merged_0000.df']


100%|██████████| 1/1 [00:03<00:00,  3.18s/it]


REMEMBER TO RECALCULATE TKI AND CHECK FV!!
Found 1 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_040015__sel_mup-data-OffBeamLight/merged_perTPC/2026_05_11_040015__sel_mup-data-OffBeamLight_merged_0000.df']


100%|██████████| 1/1 [00:02<00:00,  2.89s/it]
/tmp/ipykernel_3425744/3991466434.py:11: PerformanceWarning: indexing past lexsort depth may impact performance.
  mc_evt_df.loc[mc_evt_df.mc.iscc.isna(), ("mc", "iscc")] = 999


REMEMBER TO RECALCULATE TKI AND CHECK FV!!
evt_good rows: 12,804
mc evt: 54,287  intime evt: 96


/tmp/ipykernel_3425744/3991466434.py:16: PerformanceWarning: indexing past lexsort depth may impact performance.
  data_evt_df[("mc", "iscc")] = 999


In [30]:
perTPC_inset = 10

def perTPC_cut(df):
    in_TPC1 = (
        InFV(df.slc.vertex, det="SBND_TPC1", incathode=perTPC_inset)
        & InFV(df.mu.pfp.trk.end, det="SBND_TPC1", incathode=perTPC_inset)
        & InFV(df.p.pfp.trk.end, det="SBND_TPC1", incathode=perTPC_inset)
    )
    in_TPC2 = (
        InFV(df.slc.vertex, det="SBND_TPC2", incathode=perTPC_inset)
        & InFV(df.mu.pfp.trk.end, det="SBND_TPC2", incathode=perTPC_inset)
        & InFV(df.p.pfp.trk.end, det="SBND_TPC2", incathode=perTPC_inset)
    )
    return in_TPC1 | in_TPC2


def evt_df_fixed(df):
    slc_mudf = df.mu.pfp.trk
    slc_pdf = df.p.pfp.trk
    tki_reco = get_cc1p0pi_tki(
        slc_mudf,
        slc_pdf,
        pad_column_name(("P", "p_muon"), slc_mudf),
        pad_column_name(("P", "p_proton"), slc_pdf),
    )
    df["del_Tp_x"] = tki_reco["del_Tp_x"]
    df["del_Tp_y"] = tki_reco["del_Tp_y"]

    mc_mudf = df.mu.pfp.trk.truth.p
    mc_pdf = df.p.pfp.trk.truth.p
    tki_mc = get_cc1p0pi_tki(
        mc_mudf,
        mc_pdf,
        pad_column_name(("totp",), mc_mudf),
        pad_column_name(("totp",), mc_pdf),
    )
    df["mc_del_Tp_x"] = tki_mc["del_Tp_x"]
    df["mc_del_Tp_y"] = tki_mc["del_Tp_y"]
    df[("mc", "del_Tp_x")] = tki_mc["del_Tp_x"]
    df[("mc", "del_Tp_y")] = tki_mc["del_Tp_y"]

    n_before = len(df)
    df = df[np.abs(df.slc.vertex.x) > 10]
    if "topo_categ" not in df.columns:
        df = df.copy()
        df.loc[:, "topo_categ"] = get_topo_category(df)
    return df, n_before


mc_evt_df = mc_evt_df.loc[perTPC_cut(mc_evt_df)]
data_evt_df = data_evt_df.loc[perTPC_cut(data_evt_df)]
intime_evt_df = intime_evt_df.loc[perTPC_cut(intime_evt_df)]

for df in (mc_evt_df, data_evt_df, intime_evt_df):
    df[("mu", "pfp", "trk", "phi", "", "", "")] = np.degrees(
        np.arctan2(df[("mu", "pfp", "trk", "dir", "x", "", "")], df[("mu", "pfp", "trk", "dir", "y", "", "")])
    )
    df[("p", "pfp", "trk", "phi", "", "", "")] = np.degrees(
        np.arctan2(df[("p", "pfp", "trk", "dir", "x", "", "")], df[("p", "pfp", "trk", "dir", "y", "", "")])
    )

mc_evt_df, _ = evt_df_fixed(mc_evt_df)
data_evt_df, _ = evt_df_fixed(data_evt_df)
intime_evt_df, _ = evt_df_fixed(intime_evt_df)

print(f"perTPC selected: data={len(data_evt_df):,} mc={len(mc_evt_df):,} intime={len(intime_evt_df):,}")


/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipykernel_3425744/1776155128.py:39: PerformanceWarning: indexing past lexsort depth may impact performance.
  df[("mc", "del_Tp_x")] = tki_mc["del_Tp_x"]
/tmp/ipykernel_3425744/1776155128.py:40: PerformanceWarning: indexing past lexsort depth may impact performance.
  df[("mc", "del_Tp_y")] = tki_mc["del_Tp_y"]


perTPC selected: data=12,804 mc=54,287 intime=96


In [31]:
SYST_DISK_ROOT = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final"
CATEGORY_SUMMARY_OUT = (
    Path(PLOTS_BASE) / "syst_uncertainty_breakdown" / "final_selected" / "category_syst_summary.npz"
)
CATEGORY_SUMMARY_NPZ = category_summary_npz_path(SYST_DISK_ROOT)
if not path.isfile(CATEGORY_SUMMARY_NPZ) and CATEGORY_SUMMARY_OUT.is_file():
    CATEGORY_SUMMARY_NPZ = str(CATEGORY_SUMMARY_OUT)

import os
os.environ["NUMUCC_SYST_DISK_ROOT"] = SYST_DISK_ROOT
OVERLAY_SYST_KIND = "rate"

print("category summary:", CATEGORY_SUMMARY_NPZ, "exists =", path.isfile(CATEGORY_SUMMARY_NPZ))


category summary: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final/CategorySummary/category_syst_summary.npz exists = True


## Time-ordered cumulative slices (15 steps)


In [32]:
def split_hdr_time_chunks(hdr_df, n_splits=N_TIME_SPLITS):
    """Non-overlapping hdr chunks sorted by (run, evt) — beam_quality.ipynb logic."""
    sorted_hdr = hdr_df.sort_values(["run", "evt"], kind="mergesort")
    return [
        sorted_hdr.iloc[idx]
        for idx in np.array_split(np.arange(len(sorted_hdr)), n_splits)
    ]


def select_evt_for_hdr(evt_df, hdr_df):
    evt_base = evt_df.reset_index(level=[2])
    common = evt_base.index.intersection(hdr_df.index)
    return (
        evt_base.loc[common]
        .reset_index()
        .set_index(["__ntuple", "entry", "rec.slc..index"])
    )


def cumulative_step_slice(step_idx, hdr_splits, evt_df):
    """Return cumulative hdr/evt for step_idx in 0..n_splits-1 (chunks 0..step_idx)."""
    hdr_cum = pd.concat(hdr_splits[: step_idx + 1])
    evt_cum = select_evt_for_hdr(evt_df, hdr_cum)
    return hdr_cum, evt_cum


def format_time_ns(t_ns):
    return pd.Timestamp(t_ns, unit="ns", tz="UTC").strftime("%Y-%m-%d")


def chunk_time_label(hdr_df):
    t_min = hdr_df["global_trigger_time"].min()
    t_max = hdr_df["global_trigger_time"].max()
    return format_time_ns(t_min), format_time_ns(t_max)


def build_time_progress_context(hdr_df, hdr_splits):
    """Full-run time span and per-chunk end positions for the progress bar."""
    sorted_hdr = hdr_df.sort_values(["run", "evt"], kind="mergesort")
    t_min_ns = sorted_hdr["global_trigger_time"].min()
    t_max_ns = sorted_hdr["global_trigger_time"].max()
    span_ns = t_max_ns - t_min_ns
    if span_ns <= 0:
        raise ValueError("Expected positive global_trigger_time span")

    chunk_edge_fracs = [
        (hdr_chunk["global_trigger_time"].max() - t_min_ns) / span_ns
        for hdr_chunk in hdr_splits
    ]
    return {
        "t_min_ns": t_min_ns,
        "t_max_ns": t_max_ns,
        "span_ns": span_ns,
        "chunk_edge_fracs": chunk_edge_fracs,
    }


def time_progress_for_hdr_cum(hdr_cum, time_ctx):
    """Map cumulative hdr selection to [0, 1] progress along beam time."""
    t_current_ns = hdr_cum["global_trigger_time"].max()
    progress_frac = (t_current_ns - time_ctx["t_min_ns"]) / time_ctx["span_ns"]
    return float(np.clip(progress_frac, 0.0, 1.0)), int(t_current_ns)


def scale_mc_intime_to_hdr(hdr_cum, mc_df, intime_df):
    data_tot_pot = hdr_cum["pot"].sum() * FOM_POT_SCALE
    data_gates = hdr_cum.nbnbinfo.sum()
    mc_pot_scale = data_tot_pot / mc_tot_pot
    intime_scale = (1 - INTIME_FRACTION) * data_gates / intime_gates

    mc_scaled = mc_df.copy()
    intime_scaled = intime_df.copy()
    mc_scaled["pot_weight"] = mc_pot_scale * np.ones(len(mc_scaled))
    intime_scaled["pot_weight"] = intime_scale * np.ones(len(intime_scaled))

    data_scaled = select_evt_for_hdr(data_evt_df, hdr_cum).copy()
    data_scaled["pot_weight"] = np.ones(len(data_scaled))
    return data_scaled, mc_scaled, intime_scaled, data_tot_pot, data_gates


hdr_splits = split_hdr_time_chunks(data_hdr_df, N_TIME_SPLITS)
time_ctx = build_time_progress_context(data_hdr_df, hdr_splits)

chunk_rows = []
for i, hdr_chunk in enumerate(hdr_splits):
    start, end = chunk_time_label(hdr_chunk)
    chunk_rows.append({
        "Step": i + 1,
        "Hdr rows (this chunk)": len(hdr_chunk),
        "Start (UTC)": start,
        "End (UTC)": end,
        "POT (chunk)": hdr_chunk["pot"].sum() * FOM_POT_SCALE,
    })
chunk_df = pd.DataFrame(chunk_rows).set_index("Step")

cum_rows = []
for step_idx in range(N_TIME_SPLITS):
    hdr_cum, evt_cum = cumulative_step_slice(step_idx, hdr_splits, data_evt_df)
    start, end = chunk_time_label(hdr_cum)
    progress_frac, _ = time_progress_for_hdr_cum(hdr_cum, time_ctx)
    cum_rows.append({
        "Step": step_idx + 1,
        "Cumulative hdr rows": len(hdr_cum),
        "Cumulative evt rows": len(evt_cum),
        "Start (UTC)": start,
        "End (UTC)": end,
        "Time progress": f"{100 * progress_frac:.1f}%",
        "Cumulative POT": hdr_cum["pot"].sum() * FOM_POT_SCALE,
    })
cum_df = pd.DataFrame(cum_rows).set_index("Step")
display(chunk_df)
display(cum_df)


,Hdr rows (this chunk),Start (UTC),End (UTC),POT (chunk)
Step,,,,
1,118262,2025-02-13,2025-02-17,5.929588e+18
2,118262,2025-02-17,2025-02-21,5.972135e+18
3,118262,2025-02-21,2025-02-24,5.948053e+18
4,118262,2025-02-24,2025-02-27,5.908363e+18
5,118262,2025-02-27,2025-03-02,5.947164e+18
6,118262,2025-03-02,2025-03-06,5.943526e+18
7,118261,2025-03-06,2025-03-11,5.629916e+18
8,118261,2025-03-11,2025-03-15,5.709654e+18
9,118261,2025-03-15,2025-03-19,5.654598e+18


,Cumulative hdr rows,Cumulative evt rows,Start (UTC),End (UTC),Time progress,Cumulative POT
Step,,,,,,
1,118262,849,2025-02-13,2025-02-17,6.4%,5.929588e+18
2,236524,1720,2025-02-13,2025-02-21,12.8%,1.190172e+19
3,354786,2599,2025-02-13,2025-02-24,18.5%,1.784977e+19
4,473048,3466,2025-02-13,2025-02-27,24.4%,2.375814e+19
5,591310,4310,2025-02-13,2025-03-02,30.1%,2.970531e+19
6,709572,5168,2025-02-13,2025-03-06,36.2%,3.564883e+19
7,827833,5960,2025-02-13,2025-03-11,45.8%,4.127875e+19
8,946094,6836,2025-02-13,2025-03-15,53.0%,4.698840e+19
9,1064355,7642,2025-02-13,2025-03-19,60.1%,5.264301e+19


## Plot helpers & GIF utilities


In [33]:
def make_data_vs_mc_plotter(mc_df, intime_df, ax_ylim_ratio=AX_YLIM_RATIO):
    return partial(
        overlay_hists,
        mc_df=mc_df,
        intime_df=intime_df,
        dirt_df=None,
        ax_ylim_ratio=ax_ylim_ratio,
        ratio=True,
        textloc=[0.05, 0.68],
        approval="preliminary",
        save_fig=False,
        plot=False,
        syst=None,
        syst_kind=OVERLAY_SYST_KIND,
        syst_disk_root=SYST_DISK_ROOT,
        category_syst_summary_path=CATEGORY_SUMMARY_NPZ,
        load_syst_from_summary=True,
    )


def add_time_progress_bar(fig, progress_frac, time_ctx, t_current_ns):
    """Draw a beam-time progress bar below the overlay plot."""
    fig.subplots_adjust(bottom=0.20, top=0.90)
    ax = fig.add_axes([0.10, 0.012, 0.84, 0.075])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    bar_y, bar_h = 0.42, 0.28
    ax.add_patch(
        mpatches.Rectangle(
            (0, bar_y), 1, bar_h,
            facecolor="#ececec", edgecolor="0.45", linewidth=0.9, clip_on=False,
        )
    )
    ax.add_patch(
        mpatches.Rectangle(
            (0, bar_y), progress_frac, bar_h,
            facecolor="steelblue", edgecolor="none", alpha=0.9, clip_on=False,
        )
    )

    for edge_frac in time_ctx["chunk_edge_fracs"][:-1]:
        ax.axvline(edge_frac, ymin=bar_y - 0.08, ymax=bar_y + bar_h + 0.08,
                   color="0.55", linewidth=0.7, linestyle=":", clip_on=False)

    ax.axvline(progress_frac, ymin=bar_y - 0.18, ymax=bar_y + bar_h + 0.18,
               color="darkblue", linewidth=1.6, clip_on=False)

    ax.text(0.5, 0.92, "Beam time", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.text(0, 0.92, format_time_ns(time_ctx["t_min_ns"]), ha="left", va="bottom", fontsize=9)
    ax.text(1, 0.92, format_time_ns(time_ctx["t_max_ns"]), ha="right", va="bottom", fontsize=9)

    current_label = format_time_ns(t_current_ns)
    label_x = min(max(progress_frac, 0.06), 0.94)
    ax.text(label_x, 0.08, current_label, ha="center", va="top", fontsize=9, color="darkblue")

    pct_label = f"{100 * progress_frac:.1f}%"
    ax.text(progress_frac, bar_y + bar_h + 0.10, pct_label,
            ha="center", va="bottom", fontsize=8.5, color="darkblue")


def render_overlay_frame(
    var_config,
    plotter,
    data_df,
    mc_df,
    intime_df,
    subtitle,
    save_path,
    ax_ylim_ratio=AX_YLIM_RATIO,
    time_ctx=None,
    progress_frac=None,
    t_current_ns=None,
):
    """Single overlay_hists call; keep figure open long enough to save PNG."""
    plot_labels = [var_config.var_labels[1], "Events / Bin", subtitle]
    _orig_close = plt.close

    def _noop_close(_fig=None):
        return None

    plt.close = _noop_close
    try:
        ret = plotter(
            breakdown_type=BREAKDOWN_TYPE,
            var_config=var_config,
            plot_labels=plot_labels,
            textchi2=False,
            legend_percentages=LEGEND_PERCENTAGES_OVERRIDE.get(BREAKDOWN_TYPE),
            ax_ylim_ratio=ax_ylim_ratio,
            data_df=data_df,
            mc_df=mc_df,
            intime_df=intime_df,
        )
        fig = plt.gcf()
        if time_ctx is not None:
            add_time_progress_bar(
                fig,
                progress_frac=progress_frac,
                time_ctx=time_ctx,
                t_current_ns=t_current_ns,
            )
        fig.savefig(save_path, bbox_inches="tight", dpi=numucc_utils.dpi)
    finally:
        plt.close = _orig_close
        plt.close(fig)

    return ret


def collect_images(image_folder: Path, match=None):
    images = []
    for filepath in sorted(image_folder.iterdir()):
        if not filepath.is_file():
            continue
        if match and match not in filepath.name:
            continue
        if filepath.suffix.lower() in (".png", ".jpg", ".jpeg"):
            images.append(filepath)
    return images


def make_gif(image_paths, output_path, fps=GIF_FPS):
    if not image_paths:
        raise ValueError(f"No images found for {output_path}")
    frames = [Image.fromarray(iio.imread(p)) for p in image_paths]
    frames[0].save(
        output_path,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=1000 // fps,
        loop=0,
    )
    print(f"GIF saved to {output_path}")


## Generate frames and GIFs


In [35]:
def compute_fixed_ymax(var_config, final_step_idx=N_TIME_SPLITS - 1):
    hdr_cum, _ = cumulative_step_slice(final_step_idx, hdr_splits, data_evt_df)
    data_scaled, mc_scaled, intime_scaled, _, _ = scale_mc_intime_to_hdr(
        hdr_cum, mc_evt_df, intime_evt_df,
    )
    plotter = make_data_vs_mc_plotter(mc_scaled, intime_scaled, ax_ylim_ratio=AX_YLIM_RATIO)
    ret = plotter(
        breakdown_type=BREAKDOWN_TYPE,
        var_config=var_config,
        data_df=data_scaled,
        plot_labels=[var_config.var_labels[1], "Events / Bin", ""],
        textchi2=False,
    )
    return AX_YLIM_RATIO * np.max(ret["total_mc"])


var_configs = [
    VariableConfig.all_events(),
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.proton_momentum(),
    VariableConfig.proton_direction(),
    VariableConfig.tki_del_Tp(),
    VariableConfig.tki_del_Tp_x(),
    VariableConfig.tki_del_Tp_y(),
    VariableConfig.tki_del_p(),
    VariableConfig.tki_del_alpha(),
    VariableConfig.tki_del_phi(),
]

for var_config in var_configs:
    print(f"\n=== {var_config.var_save_name} / {BREAKDOWN_TYPE} ===")
    fixed_ymax = compute_fixed_ymax(var_config)

    with tempfile.TemporaryDirectory(prefix="fancyplot_frames_") as tmp_root:
        frame_dirs = {
            "fixed": path.join(tmp_root, "fixed_axes"),
            "auto": path.join(tmp_root, "auto_axes"),
        }
        for d in frame_dirs.values():
            makedirs(d)

        frame_paths = {"fixed": [], "auto": []}

        for step_idx in range(N_TIME_SPLITS):
            step_num = step_idx + 1
            hdr_cum, _ = cumulative_step_slice(step_idx, hdr_splits, data_evt_df)
            data_scaled, mc_scaled, intime_scaled, data_tot_pot, _ = scale_mc_intime_to_hdr(
                hdr_cum, mc_evt_df, intime_evt_df,
            )
            progress_frac, t_current_ns = time_progress_for_hdr_cum(hdr_cum, time_ctx)
            pot_str = get_pot_str(data_tot_pot)
            subtitle = ""

            plotter = make_data_vs_mc_plotter(mc_scaled, intime_scaled)
            time_kwargs = dict(
                time_ctx=time_ctx,
                progress_frac=progress_frac,
                t_current_ns=t_current_ns,
            )

            auto_path = path.join(frame_dirs["auto"], f"frame_{step_idx:02d}.png")
            ret_auto = render_overlay_frame(
                var_config, plotter, data_scaled, mc_scaled, intime_scaled,
                subtitle=subtitle, save_path=auto_path,
                ax_ylim_ratio=AX_YLIM_RATIO,
                **time_kwargs,
            )
            frame_paths["auto"].append(auto_path)

            mc_max = np.max(ret_auto["total_mc"])
            fixed_ratio = fixed_ymax / mc_max if mc_max > 0 else AX_YLIM_RATIO
            fixed_path = path.join(frame_dirs["fixed"], f"frame_{step_idx:02d}.png")
            render_overlay_frame(
                var_config, plotter, data_scaled, mc_scaled, intime_scaled,
                subtitle=subtitle, save_path=fixed_path,
                ax_ylim_ratio=fixed_ratio,
                **time_kwargs,
            )
            frame_paths["fixed"].append(fixed_path)
            print(
                f"  step {step_num:2d}: N_data={len(data_scaled):4d}  "
                f"POT={pot_str}  time={100 * progress_frac:.1f}%"
            )

        for axis_mode, paths in frame_paths.items():
            gif_name = f"{var_config.var_save_name}_{BREAKDOWN_TYPE}_cumulative_{axis_mode}_axes.gif"
            gif_path = path.join(gif_out_dir, gif_name)
            make_gif(paths, gif_path, fps=GIF_FPS)



=== integrated / topology ===
overlay_hists: could not load category_syst_summary ("Variable 'integrated' not in category summary (have: muon-dir_z, muon-p, proton-dir_z, proton-p, tki-del_Tp, tki-del_Tp_x, tki-del_Tp_y, tki-del_alpha, tki-del_p, tki-del_phi)")
no syst provided
overlay_hists: could not load category_syst_summary ("Variable 'integrated' not in category summary (have: muon-dir_z, muon-p, proton-dir_z, proton-p, tki-del_Tp, tki-del_Tp_x, tki-del_Tp_y, tki-del_alpha, tki-del_p, tki-del_phi)")
no syst provided
overlay_hists: could not load category_syst_summary ("Variable 'integrated' not in category summary (have: muon-dir_z, muon-p, proton-dir_z, proton-p, tki-del_Tp, tki-del_Tp_x, tki-del_Tp_y, tki-del_alpha, tki-del_p, tki-del_phi)")
no syst provided
  step  1: N_data= 849  POT=5.93$\times 10^{18}$  time=6.4%
overlay_hists: could not load category_syst_summary ("Variable 'integrated' not in category summary (have: muon-dir_z, muon-p, proton-dir_z, proton-p, tki-del_Tp,